# TruthLens AR — Text Baseline: XLM-RoBERTa Fake News Classifier
**Dataset:** Fakeddit (17k training split)  
**Model:** `xlm-roberta-base` fine-tuned for binary classification  
**Labels:** 0 = Real, 1 = Fake  
**Input:** `clean_title` column  

> Make sure: Runtime → Change runtime type → **T4 GPU**

## Step 1: Install dependencies

In [ ]:
!pip install transformers datasets scikit-learn --quiet

## Step 2: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 3: Load & explore the dataset

In [ ]:
import pandas as pd

TRAIN_PATH = "/content/drive/MyDrive/Colab Notebooks/final_train_17k.tsv"

df = pd.read_csv(TRAIN_PATH, sep='\t')
print(f"Total rows: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nLabel distribution (2_way_label):")
print(df['2_way_label'].value_counts())
print(f"\nMissing values in clean_title: {df['clean_title'].isna().sum()}")
df[['clean_title', '2_way_label']].head(5)

## Step 4: Preprocess — clean text & handle nulls

In [ ]:
# Keep only what we need, drop nulls
df = df[['clean_title', '2_way_label']].dropna()
df = df.rename(columns={'clean_title': 'text', '2_way_label': 'label'})

# Make sure labels are integers
df['label'] = df['label'].astype(int)

print(f"Clean dataset size: {len(df)}")
print(df['label'].value_counts())
df.head(3)

## Step 5: Train/Validation split

In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df, test_size=0.15, random_state=42, stratify=df['label']
)

print(f"Train size: {len(train_df)}")
print(f"Val size:   {len(val_df)}")
print(f"\nTrain label distribution:")
print(train_df['label'].value_counts())

## Step 6: Tokenize with XLM-RoBERTa

In [ ]:
from transformers import AutoTokenizer
import torch
from torch.utils.data import Dataset

MODEL_NAME = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_LEN = 128  # title text is short, 128 is enough

class FakeNewsDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.texts  = dataframe['text'].tolist()
        self.labels = dataframe['label'].tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = FakeNewsDataset(train_df, tokenizer, MAX_LEN)
val_dataset   = FakeNewsDataset(val_df,   tokenizer, MAX_LEN)

print(f"Sample token ids shape: {train_dataset[0]['input_ids'].shape}")
print("Tokenizer ready!")

## Step 7: Load XLM-RoBERTa model

In [ ]:
from transformers import AutoModelForSequenceClassification
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# num_labels=2 for binary: real vs fake
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
)
model = model.to(device)

BATCH_SIZE = 16

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)

print(f"Model loaded. Train batches: {len(train_loader)}")

## Step 8: Training setup — optimizer & scheduler

In [ ]:
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW

EPOCHS    = 3
LR        = 2e-5

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)

total_steps   = len(train_loader) * EPOCHS
warmup_steps  = int(0.1 * total_steps)  # 10% warmup

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print(f"Total steps: {total_steps}, Warmup steps: {warmup_steps}")

## Step 9: Train the model

In [ ]:
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score

def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += outputs.loss.item()
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average='weighted')
    return total_loss / len(loader), acc, f1


best_val_f1 = 0

for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0

    for step, batch in enumerate(train_loader):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()

        # Gradient clipping — prevents exploding gradients
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()
        total_train_loss += loss.item()

        if (step + 1) % 100 == 0:
            print(f"Epoch {epoch+1} | Step {step+1}/{len(train_loader)} | Loss: {loss.item():.4f}")

    val_loss, val_acc, val_f1 = evaluate(model, val_loader)
    avg_train_loss = total_train_loss / len(train_loader)

    print(f"\n{'='*60}")
    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | F1: {val_f1:.4f}")
    print(f"{'='*60}\n")

    # Save best model
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        model.save_pretrained('/content/drive/MyDrive/Colab Notebooks/truthlens_model')
        tokenizer.save_pretrained('/content/drive/MyDrive/Colab Notebooks/truthlens_model')
        print(f"  ✅ Best model saved! (F1: {best_val_f1:.4f})")

print("\nTraining complete!")

## Step 10: Full evaluation — classification report + confusion matrix

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Load best saved model
from transformers import AutoModelForSequenceClassification, AutoTokenizer
model = AutoModelForSequenceClassification.from_pretrained(
    '/content/drive/MyDrive/Colab Notebooks/truthlens_model'
).to(device)

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in val_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label']
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=['Real', 'Fake']))

cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Real', 'Fake'])
disp.plot(cmap='Blues')
plt.title('TruthLens — XLM-RoBERTa Confusion Matrix')
plt.savefig('/content/drive/MyDrive/Colab Notebooks/confusion_matrix.png', dpi=150)
plt.show()

## Step 11: Test on a custom sentence (try your own!)

In [ ]:
def predict(text):
    inputs = tokenizer(
        text,
        return_tensors='pt',
        max_length=128,
        truncation=True,
        padding='max_length'
    ).to(device)

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        probs   = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]
        pred    = int(probs.argmax())

    label = 'FAKE' if pred == 1 else 'REAL'
    print(f"Text  : {text}")
    print(f"Result: {label}")
    print(f"Confidence → Real: {probs[0]:.2%} | Fake: {probs[1]:.2%}")
    print(f"Credibility Score: {int(probs[0] * 100)}/100")

# Test it out
predict("PM announces new economic reform package for Bangladesh")
print()
predict("Alien spacecraft landed in Dhaka last night, government hiding the truth")

## Step 12: (Optional) Test on your own Bangla collected dataset
তোমার manually collected dataset থেকে Bangla text test করো।  
XLM-RoBERTa multilingual, তাই Bangla-তেও কাজ করবে।

In [ ]:
# তোমার নিজের dataset এর path দাও
YOUR_BANGLA_CSV = "/content/drive/MyDrive/Colab Notebooks/your_bangla_dataset.csv"

bangla_df = pd.read_csv(YOUR_BANGLA_CSV)
print(bangla_df.head())

# তোমার text column নামটা দাও নিচে
TEXT_COL = 'Claim_Text'

for _, row in bangla_df.head(5).iterrows():
    predict(str(row[TEXT_COL]))
    print('---')